In [ ]:
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import re
import warnings
from pathlib import Path
from matplotlib.lines import Line2D
import sqlite3
from scipy.optimize import least_squares
import surfinBH
import gwsurrogate as gws
import json
import sympy as sp
from IPython.display import display, Math
import lal
from tqdm.auto import tqdm
from mpl_toolkits.mplot3d import Axes3D
import precession as prececc
lal.swig_redirect_standard_output_error(False)

In [ ]:
fit = surfinBH.LoadFits('NRSur7dq4EmriRemnant')
sur_wave_precessing = gws.LoadSurrogate('NRSur7dq4')

In [ ]:
def entropy(M, J):
    M = np.asarray(M)
    J = np.asarray(J)
    arg_sqrt = M**2 - ((J**2)/(M**2))
    arg_sqrt = np.where(arg_sqrt < 0, np.nan, arg_sqrt)
    S = 2 * np.pi * M * (M + np.sqrt(arg_sqrt))
    
    return S

In [ ]:
def get_0PN_precessing(
    a_prec,
    th1,
    th2,
    dphi,
    chi1,
    chi2,
    m1,
    m2,
    c=1.0,
    G=1.0,
    separation_is_dimensionless=True
):
   
    def to_float_scalar(value, name):
        try:
            return float(value)
        except Exception as exc:
            raise TypeError(
                f"{name} must be a float, but {value!r} was given."
            ) from exc

    chi1 = to_float_scalar(chi1, "chi1")
    chi2 = to_float_scalar(chi2, "chi2")
    m1 = to_float_scalar(m1, "m1")
    m2 = to_float_scalar(m2, "m2")
    c = to_float_scalar(c, "c")
    G = to_float_scalar(G, "G")

    if m1 <= 0.0 or m2 <= 0.0:
        raise ValueError("m1 and m2 must be positive.")
    if c <= 0.0:
        raise ValueError("c must be positive.")
    if G <= 0.0:
        raise ValueError("G must be positive.")
    if abs(chi1) > 1.0 or abs(chi2) > 1.0:
        raise ValueError("Dimensionless spins must satisfy |chi_i| <= 1.")

    a_prec = np.asarray(a_prec, dtype=float)
    th1 = np.asarray(th1, dtype=float)
    th2 = np.asarray(th2, dtype=float)
    dphi = np.asarray(dphi, dtype=float)

    a_prec, th1, th2, dphi = np.broadcast_arrays(
        a_prec,
        th1,
        th2,
        dphi
    )

    if np.any(a_prec <= 0.0):
        raise ValueError("All separation values must be positive.")

    m = m1 + m2
    mu = m1 * m2 / m
    nu = mu / m
    q_prec = m2 / m1

    if separation_is_dimensionless:
        Gamma = G / a_prec
    else:
        Gamma = G * m / a_prec

    sqrt_Gamma = np.sqrt(Gamma)
    gamma_Blanchet = Gamma / c**2

    E_bind_0PN = -0.5 * mu * Gamma
    E_bind = E_bind_0PN

    E_tot = m + E_bind / c**2

    L_orb_0PN = G * mu * m / sqrt_Gamma
    L_orb = L_orb_0PN

    L_vec = np.stack(
        [
            np.zeros_like(L_orb),
            np.zeros_like(L_orb),
            L_orb
        ],
        axis=-1
    )

    S1_mag = G * chi1 * m1**2
    S2_mag = G * chi2 * m2**2

    S1_vec = np.stack(
        [
            S1_mag * np.sin(th1),
            np.zeros_like(th1),
            S1_mag * np.cos(th1)
        ],
        axis=-1
    )

    S2_vec = np.stack(
        [
            S2_mag * np.sin(th2) * np.cos(dphi),
            S2_mag * np.sin(th2) * np.sin(dphi),
            S2_mag * np.cos(th2)
        ],
        axis=-1
    )

    S_tot_vec = S1_vec + S2_vec

    J_vec = L_vec + S_tot_vec / c
    J_tot = np.sqrt(np.sum(J_vec**2, axis=-1))

    return {
        "q_prec": q_prec,

        "Gamma": Gamma,
        "gamma_Blanchet": gamma_Blanchet,

        "nu": nu,
        "mu": mu,
        "m": m,
        "m1": m1,
        "m2": m2,

        "E_bind": E_bind,
        "E_tot": E_tot,
        "E_bind_0PN": E_bind_0PN,

        "L_orb": L_orb,
        "L_orb_0PN": L_orb_0PN,
        "L_vec": L_vec,

        "S1_vec": S1_vec,
        "S2_vec": S2_vec,
        "S_tot_vec": S_tot_vec,

        "J_vec": J_vec,
        "J_tot": J_tot
    }

In [ ]:
def get_1PN_precessing(
    a_prec,
    th1,
    th2,
    dphi,
    chi1,
    chi2,
    m1,
    m2,
    c=1.0,
    G=1.0,
    separation_is_dimensionless=True
):
    
    def to_float_scalar(value, name):
        try:
            return float(value)
        except Exception as exc:
            raise TypeError(
                f"{name} must be a float, but {value!r} was given."
            ) from exc

    chi1 = to_float_scalar(chi1, "chi1")
    chi2 = to_float_scalar(chi2, "chi2")
    m1 = to_float_scalar(m1, "m1")
    m2 = to_float_scalar(m2, "m2")
    c = to_float_scalar(c, "c")
    G = to_float_scalar(G, "G")

    if m1 <= 0.0 or m2 <= 0.0:
        raise ValueError("m1 and m2 must be positive.")
    if c <= 0.0:
        raise ValueError("c must be positive.")
    if G <= 0.0:
        raise ValueError("G must be positive.")
    if abs(chi1) > 1.0 or abs(chi2) > 1.0:
        raise ValueError("Dimensionless spins must satisfy |chi_i| <= 1.")

    a_prec = np.asarray(a_prec, dtype=float)
    th1 = np.asarray(th1, dtype=float)
    th2 = np.asarray(th2, dtype=float)
    dphi = np.asarray(dphi, dtype=float)

    a_prec, th1, th2, dphi = np.broadcast_arrays(
        a_prec,
        th1,
        th2,
        dphi
    )

    if np.any(a_prec <= 0.0):
        raise ValueError("All separation values must be positive.")

    m = m1 + m2
    mu = m1 * m2 / m
    nu = mu / m

    if separation_is_dimensionless:
        Gamma = G / a_prec
    else:
        Gamma = G * m / a_prec

    sqrt_Gamma = np.sqrt(Gamma)
    gamma_Blanchet = Gamma / c**2

    E_bind_0PN = -0.5 * mu * Gamma

    E_bind_1PN = (
        mu
        * Gamma**2
        * (7.0 - nu)
        / 8.0
        / c**2
    )

    E_bind = (
        E_bind_0PN
        +
        E_bind_1PN
    )

    E_tot = m + E_bind / c**2

    L_orb_0PN = (
        G
        * mu
        * m
        / sqrt_Gamma
    )

    L_orb_1PN = (
        2.0
        * G
        * mu
        * m
        * sqrt_Gamma
        / c**2
    )

    L_orb = (
        L_orb_0PN
        +
        L_orb_1PN
    )

    L_vec = np.stack(
        [
            np.zeros_like(L_orb),
            np.zeros_like(L_orb),
            L_orb
        ],
        axis=-1
    )

    S1_mag = G * chi1 * m1**2
    S2_mag = G * chi2 * m2**2

    S1_vec = np.stack(
        [
            S1_mag * np.sin(th1),
            np.zeros_like(th1),
            S1_mag * np.cos(th1)
        ],
        axis=-1
    )

    S2_vec = np.stack(
        [
            S2_mag * np.sin(th2) * np.cos(dphi),
            S2_mag * np.sin(th2) * np.sin(dphi),
            S2_mag * np.cos(th2)
        ],
        axis=-1
    )

    S_tot_vec = S1_vec + S2_vec

    J_vec = L_vec + S_tot_vec / c
    J_tot = np.sqrt(np.sum(J_vec**2, axis=-1))

    return {
        "Gamma": Gamma,
        "gamma_Blanchet": gamma_Blanchet,

        "nu": nu,
        "mu": mu,
        "m": m,
        "m1": m1,
        "m2": m2,

        "E_bind": E_bind,
        "E_tot": E_tot,

        "E_bind_0PN": E_bind_0PN,
        "E_bind_1PN": E_bind_1PN,

        "L_orb": L_orb,
        "L_orb_0PN": L_orb_0PN,
        "L_orb_1PN": L_orb_1PN,

        "L_vec": L_vec,

        "S1_vec": S1_vec,
        "S2_vec": S2_vec,
        "S_tot_vec": S_tot_vec,

        "J_vec": J_vec,
        "J_tot": J_tot
    }

In [ ]:
def get_2PN_precessing(
    a_prec,
    th1,
    th2,
    dphi,
    chi1,
    chi2,
    m1,
    m2,
    c=1.0,
    G=1.0,
    separation_is_dimensionless=True
):
    

    def to_float_scalar(value, name):
        try:
            return float(value)
        except Exception as exc:
            raise TypeError(
                f"{name} must be a float, but {value!r} was given."
            ) from exc

    chi1 = to_float_scalar(chi1, "chi1")
    chi2 = to_float_scalar(chi2, "chi2")
    m1 = to_float_scalar(m1, "m1")
    m2 = to_float_scalar(m2, "m2")
    c = to_float_scalar(c, "c")
    G = to_float_scalar(G, "G")

    if m1 <= 0.0 or m2 <= 0.0:
        raise ValueError("m1 and m2 must be positive.")

    if c <= 0.0:
        raise ValueError("c must be positive.")

    if G <= 0.0:
        raise ValueError("G must be positive.")

    if abs(chi1) > 1.0 or abs(chi2) > 1.0:
        raise ValueError("Dimensionless Kerr spins must satisfy |chi_i| <= 1.")

    CQ1 = 1.0
    CQ2 = 1.0

    a_prec = np.asarray(a_prec, dtype=float)
    th1 = np.asarray(th1, dtype=float)
    th2 = np.asarray(th2, dtype=float)
    dphi = np.asarray(dphi, dtype=float)

    a_prec, th1, th2, dphi = np.broadcast_arrays(
        a_prec,
        th1,
        th2,
        dphi
    )

    if np.any(a_prec <= 0.0):
        raise ValueError("All separation values must be positive.")

    m = m1 + m2
    mu = m1 * m2 / m
    nu = mu / m

    if separation_is_dimensionless:

        Gamma = G / a_prec
    else:

        Gamma = G * m / a_prec

    sqrt_Gamma = np.sqrt(Gamma)

    S1_mag = G * chi1 * m1**2
    S2_mag = G * chi2 * m2**2

    S1n = S1_mag * np.sin(th1)
    S1lam = np.zeros_like(th1)
    S1L = S1_mag * np.cos(th1)

    S2n = S2_mag * np.sin(th2) * np.cos(dphi)
    S2lam = S2_mag * np.sin(th2) * np.sin(dphi)
    S2L = S2_mag * np.cos(th2)

    S1_vec = np.stack(
        [S1n, S1lam, S1L],
        axis=-1
    )

    S2_vec = np.stack(
        [S2n, S2lam, S2L],
        axis=-1
    )

    S_tot_vec = S1_vec + S2_vec

    S1S2 = np.sum(S1_vec * S2_vec, axis=-1)
    S1sq = np.sum(S1_vec * S1_vec, axis=-1)
    S2sq = np.sum(S2_vec * S2_vec, axis=-1)

    bracket_SO_L = (
        S1L * (4.0 * m1 * m2 + 3.0 * m2**2)
        +
        S2L * (3.0 * m1**2 + 4.0 * m1 * m2)
    )

    bracket_SS_12 = (
        -S1S2
        + 3.0 * S1L * S2L
    )

    bracket_SS_11 = (
        -S1sq
        + 3.0 * S1L**2
    )

    bracket_SS_22 = (
        -S2sq
        + 3.0 * S2L**2
    )

    bracket_SS_self = (
        CQ1 * (m2 / m1) * bracket_SS_11
        +
        CQ2 * (m1 / m2) * bracket_SS_22
    )

    bracket_SS_total = (
        bracket_SS_12
        +
        0.5 * bracket_SS_self
    )

    E_bind_0PN = (
        -0.5 * mu * Gamma
    )

    E_bind_1PN_NS = (
        mu * Gamma**2 * (7.0 - nu) / 8.0
    )

    E_bind_SO_15PN = (
        -mu * Gamma**2.5
        / (3.0 * G * m**2 * m1 * m2)
        * bracket_SO_L
    )

    E_bind_2PN_NS = (
        mu * Gamma**3 * (7.0 - 49.0 * nu - nu**2) / 16.0
    )

    E_bind_SS_12_2PN = (
        -Gamma**3
        / (2.0 * G**2 * m**3)
        * bracket_SS_12
    )

    E_bind_SS_self_2PN = (
        -Gamma**3
        / (4.0 * G**2 * m**3)
        * bracket_SS_self
    )

    E_bind_SS_2PN = (
        E_bind_SS_12_2PN
        +
        E_bind_SS_self_2PN
    )

    E_bind = (
        E_bind_0PN
        +
        E_bind_1PN_NS / c**2
        +
        E_bind_SO_15PN / c**3
        +
        (
            E_bind_2PN_NS
            +
            E_bind_SS_2PN
        ) / c**4
    )

    M_tot = m + E_bind / c**2

    E_tot_energy = m * c**2 + E_bind

    L_NS_0PN = (
        G * mu * m / sqrt_Gamma
    )

    L_NS_1PN = (
        2.0 * G * mu * m * sqrt_Gamma
    )

    L_NS_2PN = (
        G * mu * m * Gamma**1.5 * (5.0 - 9.0 * nu) / 2.0
    )


    L_SO_ell_15PN = (
        -5.0 * Gamma
        / (6.0 * m**2)
        * bracket_SO_L
    )

    L_SO_n_15PN = (
        Gamma
        / (2.0 * m**2)
        * (
            m2**2 * S1n
            +
            m1**2 * S2n
        )
    )

    L_SO_lam_15PN = (
        -Gamma
        / m**2
        * (
            m2 * (2.0 * m1 + m2) * S1lam
            +
            m1 * (m1 + 2.0 * m2) * S2lam
        )
    )

    L_SS_ell_2PN = (
        -Gamma**1.5
        / (G * m**2)
        * bracket_SS_total
    )

    L_SS_n_2PN = np.zeros_like(Gamma)
    L_SS_lam_2PN = np.zeros_like(Gamma)


    L_ell = (
        L_NS_0PN
        +
        L_NS_1PN / c**2
        +
        L_SO_ell_15PN / c**3
        +
        (
            L_NS_2PN
            +
            L_SS_ell_2PN
        ) / c**4
    )

    L_n = (
        L_SO_n_15PN / c**3
        +
        L_SS_n_2PN / c**4
    )

    L_lam = (
        L_SO_lam_15PN / c**3
        +
        L_SS_lam_2PN / c**4
    )

    L_vec = np.stack(
        [L_n, L_lam, L_ell],
        axis=-1
    )


    J_vec = L_vec + S_tot_vec / c

    J_tot = np.sqrt(
        np.sum(J_vec**2, axis=-1)
    )


    return {
        "Gamma": Gamma,
        "nu": nu,
        "mu": mu,
        "m": m,
        "m1": m1,
        "m2": m2,

        "CQ1": CQ1,
        "CQ2": CQ2,

        "E_bind": E_bind,
        "M_tot": M_tot,
        "E_tot_energy": E_tot_energy,

        "E_tot": M_tot,

        "E_bind_0PN": E_bind_0PN,
        "E_bind_1PN_NS": E_bind_1PN_NS / c**2,
        "E_bind_SO_15PN": E_bind_SO_15PN / c**3,
        "E_bind_2PN_NS": E_bind_2PN_NS / c**4,
        "E_bind_SS_2PN": E_bind_SS_2PN / c**4,
        "E_bind_SS_12_2PN": E_bind_SS_12_2PN / c**4,
        "E_bind_SS_self_2PN": E_bind_SS_self_2PN / c**4,

        "L_vec": L_vec,
        "L_n": L_n,
        "L_lam": L_lam,
        "L_ell": L_ell,

        "L_NS_0PN": L_NS_0PN,
        "L_NS_1PN": L_NS_1PN / c**2,
        "L_NS_2PN": L_NS_2PN / c**4,

        "L_SO_n_15PN": L_SO_n_15PN / c**3,
        "L_SO_lam_15PN": L_SO_lam_15PN / c**3,
        "L_SO_ell_15PN": L_SO_ell_15PN / c**3,

        "L_SS_ell_2PN": L_SS_ell_2PN / c**4,
        "L_SS_n_2PN": L_SS_n_2PN / c**4,
        "L_SS_lam_2PN": L_SS_lam_2PN / c**4,

        "S1_vec": S1_vec,
        "S2_vec": S2_vec,
        "S_tot_vec": S_tot_vec,

        "S1n": S1n,
        "S1lam": S1lam,
        "S1L": S1L,
        "S2n": S2n,
        "S2lam": S2lam,
        "S2L": S2L,

        "S1S2": S1S2,
        "S1sq": S1sq,
        "S2sq": S2sq,

        "bracket_SO_L": bracket_SO_L,
        "bracket_SS_12": bracket_SS_12,
        "bracket_SS_11": bracket_SS_11,
        "bracket_SS_22": bracket_SS_22,
        "bracket_SS_self": bracket_SS_self,
        "bracket_SS_total": bracket_SS_total,

        "J_vec": J_vec,
        "J_tot": J_tot,
    }